# Detecting Water Leaks Using Acoustic Signals
### Community Engagement Project (CEP) - T.Y. B.Sc. IT (Sem V)
**Student:** Jayprakash Shyamdev Maurya (Roll No: 24302A0019)  
**College:** Vidyalankar School of Information Technology (VSIT), Mumbai  
**Industry Partner:** Precision Grow  
**SDG Alignment:** SDG 6 (Clean Water and Sanitation)

This notebook demonstrates the end-to-end acoustic signal processing and machine learning pipeline for detecting pipe leaks.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import joblib

# Ensure project root is accessible
PROJECT_ROOT = os.path.dirname(os.path.abspath(''))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from preprocessing.preprocess import load_signal, validate_signal, bandpass_filter, normalize, segment
from feature_extraction.features import extract_features_from_signal, FEATURE_NAMES

print("Modules imported successfully!")

## 1. Load and Compare Normal vs. Leak Acoustic Signals

In [ ]:
normal_path = os.path.join(PROJECT_ROOT, 'dataset', 'normal', 'normal_001.wav')
leak_path = os.path.join(PROJECT_ROOT, 'dataset', 'leak', 'leak_001.wav')

sig_norm, sr = load_signal(normal_path)
sig_leak, _ = load_signal(leak_path)

t = np.linspace(0, len(sig_norm)/sr, len(sig_norm))

plt.figure(figsize=(12, 5))
plt.subplot(2, 1, 1)
plt.plot(t, sig_norm, color='tab:blue', lw=0.8)
plt.title("Normal Pipeline Sound (Flow + 50Hz Hum + Rumble)")
plt.ylabel("Amplitude")
plt.grid(True, linestyle="--", alpha=0.5)

plt.subplot(2, 1, 2)
plt.plot(t, sig_leak, color='tab:red', lw=0.8)
plt.title("Possible Leak Sound (Turbulent Hiss + Bursts in 500-3000 Hz Band)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 2. Frequency-Domain Analysis (FFT Spectrum)

In [ ]:
fft_norm = np.abs(np.fft.rfft(sig_norm))
fft_leak = np.abs(np.fft.rfft(sig_leak))
freqs = np.fft.rfftfreq(len(sig_norm), 1.0/sr)

plt.figure(figsize=(10, 4))
plt.plot(freqs, fft_norm, label='Normal (Low Frequency Dominant)', color='tab:blue', alpha=0.8)
plt.plot(freqs, fft_leak, label='Leak (High Energy in 800-2400 Hz)', color='tab:red', alpha=0.8)
plt.title("FFT Frequency Spectrum Comparison")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.xlim(0, 4000)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## 3. Feature Extraction and ML Inference

In [ ]:
model_artifact = joblib.load(os.path.join(PROJECT_ROOT, 'model', 'leak_model.joblib'))
model = model_artifact['best_model']

df_features = extract_features_from_signal(sig_leak, sr=sr)
X = df_features[FEATURE_NAMES].values
leak_probs = model.predict_proba(X)[:, 1]

print("Extracted Features Shape:", df_features.shape)
print("Segment-wise Leak Probabilities:", [f"{p*100:.1f}%" for p in leak_probs])
print(f"Mean Leak Probability: {np.mean(leak_probs)*100:.1f}%")

if np.mean(leak_probs) >= 0.50:
    print("\nRESULT: POSSIBLE WATER LEAK DETECTED (High Confidence)")
else:
    print("\nRESULT: NORMAL CONDITION")